<a href="https://colab.research.google.com/github/virajrank48-max/big_data/blob/main/Fraud_Detection_Modeling_Big_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Full-Dataset Credit Card Fraud Detection

This notebook applies Spark ML directly to the complete IBM credit-card transaction dataset. It is designed to avoid the sampling bias and memory crashes caused by converting millions of rows to Pandas and repeatedly running SMOTE, ADASYN, NearMiss, and undersampling pipelines.

The default run trains one weighted Random Forest on all valid transactions. Class weights address the severe fraud imbalance without duplicating rows or collecting the dataset on the driver. An optional Spark GBT comparison is provided separately; it is disabled by default to keep the main run within a practical Colab time budget.

**Important:** A guaranteed 5–10 minute runtime depends on the Colab tier, input format, storage speed, Spark partitions, and dataset size. This notebook removes the main avoidable bottlenecks, but no notebook can guarantee a fixed runtime on shared hardware.

## 1. Runtime

This project used a Colab runtime with sufficient RAM and CPU cores. The model reads Parquet when available because it is substantially faster and lighter than repeatedly parsing the raw CSV. The raw CSV remains a fallback.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install -q -U "xgboost>=2.0,<4"

In [ ]:

import os
import time
import pandas as pd
from pyspark.sql import SparkSession, functions as F
from pyspark.storagelevel import StorageLevel
from xgboost.spark import SparkXGBClassifier
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.functions import vector_to_array
import subprocess
import xgboost as xgb

In [ ]:
RANDOM_STATE = 42
TEST_FRACTION = 0.20
SHUFFLE_PARTITIONS = max(32, min(128, (os.cpu_count() or 4) * 4))
USE_GPU = True
GPU_DEVICE = 'cuda'
XGB_NUM_WORKERS = 1
XGB_NUM_TREES = 80
XGB_MAX_DEPTH = 8
XGB_LEARNING_RATE = 0.10

PARQUET_PATH = '/content/drive/MyDrive/cleaned_transactions'
CSV_PATH = '/content/extracted_folder/credit_card_transactions-ibm_v2.csv'

spark = (
    SparkSession.builder
    .appName('FullDatasetFraudDetection')
    .config('spark.sql.adaptive.enabled', 'true')
    .config('spark.sql.adaptive.coalescePartitions.enabled', 'true')
    .config('spark.sql.shuffle.partitions', str(SHUFFLE_PARTITIONS))
    .config('spark.serializer', 'org.apache.spark.serializer.KryoSerializer')
    .config('spark.driver.maxResultSize', '2g')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print('Spark:', spark.version)
print('Shuffle partitions:', SHUFFLE_PARTITIONS)
print('Cores:', spark.sparkContext.defaultParallelism)


Spark: 4.0.4
Shuffle partitions: 32
Cores: 2


## 1. Install and verify GPU dependencies

The GPU path uses XGBoost 2.x or newer through `xgboost.spark`. Spark's readers and SQL transformations remain CPU operations, but the XGBoost tree-building stage runs on the CUDA device. If the runtime does not expose a GPU, fail early instead of silently running a CPU model.


In [ ]:


if USE_GPU:
    gpu_check = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
        capture_output=True,
        text=True,
    )
    if gpu_check.returncode != 0:
        raise RuntimeError(
            'CUDA GPU is unavailable. In Colab, select Runtime > Change runtime type > T4 GPU, then restart and rerun all cells.'
        )
    print('CUDA device:', gpu_check.stdout.strip())
    print('XGBoost:', xgb.__version__)
else:
    print('GPU path disabled; the model will run on CPU.')

CUDA device: Tesla T4, 15360 MiB
XGBoost: 3.4.1


## 2. Read and prepare the full dataset

Only compact, useful predictors are retained. High-cardinality identifiers such as `User`, `Card`, `Merchant Name`, and merchant location are excluded because one-hot encoding them across 24 million rows is a major source of memory pressure and usually creates memorization rather than generalizable fraud signals.

In [ ]:
required_columns = {
    'Is Fraud?', 'Amount', 'Year', 'Month', 'Day', 'Time',
    'Use Chip', 'MCC', 'Errors?'
}

read_start = time.perf_counter()
if os.path.isdir(PARQUET_PATH):
    source_df = spark.read.parquet(PARQUET_PATH)
    source_name = 'Parquet'
elif os.path.isfile(CSV_PATH):
    source_df = (
        spark.read
        .option('header', True)
        .option('inferSchema', True)
        .option('mode', 'PERMISSIVE')
        .csv(CSV_PATH)
    )
    source_name = 'CSV'
else:
    raise FileNotFoundError('Neither PARQUET_PATH nor CSV_PATH exists.')

missing_columns = required_columns.difference(source_df.columns)
if missing_columns:
    raise ValueError(f'Missing required columns: {sorted(missing_columns)}')

# Safely extract hour whether 'Time' is already a TimestampType or a StringType
time_col = F.col('Time')
if dict(source_df.dtypes).get('Time') == 'string':
    # If it is a string containing only 'HH:mm:ss'
    hour_expr = F.hour(F.to_timestamp(F.trim(time_col), 'HH:mm:ss'))
else:
    # If it is already a timestamp or datetime-formatted string
    hour_expr = F.hour(time_col)

prepared = (
    source_df.select(*sorted(required_columns))
    .withColumn('Amount', F.regexp_replace(F.col('Amount').cast('string'), r'[$,]', '').cast('double'))
    .withColumn('Year', F.col('Year').cast('double'))
    .withColumn('Month', F.col('Month').cast('double'))
    .withColumn('Day', F.col('Day').cast('double'))
    .withColumn('MCC', F.col('MCC').cast('double'))
    .withColumn('transaction_hour', hour_expr.cast('double'))
    .withColumn(
        'fraud_label',
        F.when(F.lower(F.trim(F.col('Is Fraud?').cast('string'))) == 'yes', F.lit(1.0))
         .when(F.lower(F.trim(F.col('Is Fraud?').cast('string'))) == 'no', F.lit(0.0))
    )
    .withColumn('Use Chip', F.coalesce(F.trim(F.col('Use Chip').cast('string')), F.lit('Unknown')))
    .withColumn('Errors?', F.coalesce(F.trim(F.col('Errors?').cast('string')), F.lit('Unknown')))
    .drop('Is Fraud?', 'Time')
    .dropna(subset=['Amount', 'Year', 'Month', 'Day', 'MCC', 'transaction_hour', 'fraud_label'])
    .repartition(SHUFFLE_PARTITIONS)
)

# Disk-backed persistence prevents the full dataset from exhausting RAM.
prepared = prepared.persist(StorageLevel.DISK_ONLY)
prepared_count = prepared.count()
print(f'Source: {source_name}')
print(f'Prepared full-dataset rows: {prepared_count:,}')
print(f'Read and preparation time: {time.perf_counter() - read_start:.1f}s')
prepared.groupBy('fraud_label').count().orderBy('fraud_label').show()

Source: Parquet
Prepared full-dataset rows: 24,386,900
Read and preparation time: 2.4s
+-----------+--------+
|fraud_label|   count|
+-----------+--------+
|        0.0|24357143|
|        1.0|   29757|
+-----------+--------+



## 3.train/test split

The split is performed on the complete prepared dataset. No rows are copied for oversampling, and the test set remains untouched. Class weights are computed from the training counts and applied only during fitting.

In [ ]:
split_start = time.perf_counter()
train_data, test_data = prepared.randomSplit(
    [1.0 - TEST_FRACTION, TEST_FRACTION],
    seed=RANDOM_STATE
)

train_counts = {
    float(row['fraud_label']): int(row['count'])
    for row in train_data.groupBy('fraud_label').count().collect()
}
n_normal = train_counts.get(0.0, 0)
n_fraud = train_counts.get(1.0, 0)
if not n_normal or not n_fraud:
    raise ValueError(f'Both classes are required; observed counts: {train_counts}')

total_train = n_normal + n_fraud
normal_weight = total_train / (2.0 * n_normal)
fraud_weight = total_train / (2.0 * n_fraud)

train_data = train_data.withColumn(
    'class_weight',
    F.when(F.col('fraud_label') == 1.0, F.lit(fraud_weight)).otherwise(F.lit(normal_weight))
)

print(f'Train normal: {n_normal:,}; train fraud: {n_fraud:,}')
print(f'Class weights: normal={normal_weight:.4f}, fraud={fraud_weight:.4f}')
print(f'Split preparation time: {time.perf_counter() - split_start:.1f}s')


Train normal: 19,484,114; train fraud: 23,877
Class weights: normal=0.5006, fraud=408.5101
Split preparation time: 37.8s


## 4. Train the full-dataset GPU XGBoost model

Spark ML's `RandomForestClassifier` is CPU-only. The default model below uses `xgboost.spark.SparkXGBClassifier` with `device='cuda'`, so the complete training dataset stays distributed while XGBoost uses the Colab GPU for tree construction. With one Colab GPU, keep `num_workers=1`; multiple workers require a multi-GPU Spark cluster.


In [ ]:


numeric_columns = ['Year', 'Month', 'Day', 'Amount', 'MCC', 'transaction_hour']
categorical_columns = ['Use Chip', 'Errors?']

indexers = [
    StringIndexer(inputCol=column, outputCol=f'{column}_index', handleInvalid='keep')
    for column in categorical_columns
]
assembler = VectorAssembler(
    inputCols=numeric_columns + [f'{column}_index' for column in categorical_columns],
    outputCol='features',
    handleInvalid='skip'
)

xgb = SparkXGBClassifier(
    features_col='features',
    label_col='fraud_label',
    weight_col='class_weight',
    prediction_col='prediction',
    probability_col='probability',
    raw_prediction_col='rawPrediction',
    num_workers=XGB_NUM_WORKERS,
    device=GPU_DEVICE if USE_GPU else 'cpu',
    tree_method='hist',
    n_estimators=XGB_NUM_TREES,
    max_depth=XGB_MAX_DEPTH,
    learning_rate=XGB_LEARNING_RATE,
    subsample=0.85,
    colsample_bytree=0.85,
    min_child_weight=2.0,
    reg_lambda=1.0,
    eval_metric='logloss',
    random_state=RANDOM_STATE,

)

xgb_pipeline = Pipeline(stages=indexers + [assembler, xgb])
train_start = time.perf_counter()
xgb_model = xgb_pipeline.fit(train_data)
print(f'XGBoost training time: {time.perf_counter() - train_start:.1f}s')
print('Training device:', GPU_DEVICE if USE_GPU else 'cpu')


INFO:XGBoost-PySpark:Running xgboost-3.4.1 on 1 workers with
	booster params: {'objective': 'binary:logistic', 'colsample_bytree': 0.85, 'device': 'cuda', 'eval_metric': 'logloss', 'learning_rate': 0.1, 'max_depth': 8, 'min_child_weight': 2.0, 'random_state': 42, 'reg_lambda': 1.0, 'subsample': 0.85, 'tree_method': 'hist', 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 80}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
INFO:XGBoost-PySpark:Finished xgboost training!


XGBoost training time: 312.4s
Training device: cuda


## 5. Evaluate on the complete held-out test set

PR-AUC, precision, recall, and F1-score for the fraud class are more informative than accuracy for this highly imbalanced problem.

In [ ]:


predictions = (
    xgb_model.transform(test_data)
    .select('fraud_label', 'prediction', 'rawPrediction', 'probability')
    .withColumn('fraud_probability', vector_to_array('probability')[1])
    .persist(StorageLevel.DISK_ONLY)
)

# Materialize once; subsequent metric actions reuse disk-backed predictions.
prediction_count = predictions.count()
roc_auc = BinaryClassificationEvaluator(
    labelCol='fraud_label', rawPredictionCol='rawPrediction', metricName='areaUnderROC'
).evaluate(predictions)
pr_auc = BinaryClassificationEvaluator(
    labelCol='fraud_label', rawPredictionCol='rawPrediction', metricName='areaUnderPR'
).evaluate(predictions)

cm = predictions.agg(
    F.sum(F.when((F.col('fraud_label') == 1.0) & (F.col('prediction') == 1.0), 1).otherwise(0)).alias('TP'),
    F.sum(F.when((F.col('fraud_label') == 0.0) & (F.col('prediction') == 1.0), 1).otherwise(0)).alias('FP'),
    F.sum(F.when((F.col('fraud_label') == 0.0) & (F.col('prediction') == 0.0), 1).otherwise(0)).alias('TN'),
    F.sum(F.when((F.col('fraud_label') == 1.0) & (F.col('prediction') == 0.0), 1).otherwise(0)).alias('FN')
).first()

tp, fp, tn, fn = [int(cm[column] or 0) for column in ['TP', 'FP', 'TN', 'FN']]
precision = tp / (tp + fp) if tp + fp else 0.0
recall = tp / (tp + fn) if tp + fn else 0.0
f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0

print(f'Test rows: {prediction_count:,}')
print(f'ROC-AUC: {roc_auc:.4f}')
print(f'PR-AUC: {pr_auc:.4f}')
print(f'TP={tp:,}  FP={fp:,}  TN={tn:,}  FN={fn:,}')
print(f'Fraud precision: {precision:.4f}')
print(f'Fraud recall: {recall:.4f}')
print(f'Fraud F1-score: {f1:.4f}')


Test rows: 4,878,909
ROC-AUC: 0.9788
PR-AUC: 0.3989
TP=5,322  FP=291,174  TN=4,581,855  FN=558
Fraud precision: 0.0179
Fraud recall: 0.9051
Fraud F1-score: 0.0352


## 6. Select an operating threshold

The default Spark threshold is 0.5. For fraud detection, the threshold should be selected according to the business cost of missed fraud versus false alarms. The following distributed calculation reports F1, precision, and recall without collecting predictions.

In [ ]:
threshold_rows = []
for threshold in [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]:
    counts = predictions.agg(
        F.sum(F.when((F.col('fraud_label') == 1.0) & (F.col('fraud_probability') >= threshold), 1).otherwise(0)).alias('TP'),
        F.sum(F.when((F.col('fraud_label') == 0.0) & (F.col('fraud_probability') >= threshold), 1).otherwise(0)).alias('FP'),
        F.sum(F.when((F.col('fraud_label') == 1.0) & (F.col('fraud_probability') < threshold), 1).otherwise(0)).alias('FN')
    ).first()
    tp_t, fp_t, fn_t = [int(counts[column] or 0) for column in ['TP', 'FP', 'FN']]
    precision_t = tp_t / (tp_t + fp_t) if tp_t + fp_t else 0.0
    recall_t = tp_t / (tp_t + fn_t) if tp_t + fn_t else 0.0
    f1_t = 2 * precision_t * recall_t / (precision_t + recall_t) if precision_t + recall_t else 0.0
    threshold_rows.append((threshold, precision_t, recall_t, f1_t))

threshold_results = spark.createDataFrame(
    threshold_rows, ['threshold', 'precision', 'recall', 'f1']
)
threshold_results.orderBy(F.desc('f1')).show(truncate=False)


+---------+---------------------+------------------+---------------------+
|threshold|precision            |recall            |f1                   |
+---------+---------------------+------------------+---------------------+
|0.9      |0.12057535579613186  |0.6743197278911565|0.20457125167681356  |
|0.8      |0.058195381156754544 |0.7748299319727892|0.10825967113392262  |
|0.7      |0.039658970384270215 |0.8267006802721089|0.07568703775788245  |
|0.6      |0.026329252600627354 |0.8664965986394558|0.051105616602554776 |
|0.5      |0.01794965193459608  |0.9051020408163265|0.035201206444955946 |
|0.4      |0.012330059998651716 |0.9331632653061225|0.024338530461975205 |
|0.3      |0.008662423133529364 |0.9549319727891157|0.017169101121877688 |
|0.2      |0.0058758940574043855|0.9738095238095238|0.011681303997470341 |
|0.1      |0.0037323687857116774|0.9908163265306122|0.0074367237058013715|
+---------+---------------------+------------------+---------------------+



# Logistic Regression



In [ ]:


logistic_regression = LogisticRegression(
    labelCol="fraud_label",
    featuresCol="features",
    weightCol="class_weight",
    predictionCol="prediction",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction",
    maxIter=50,
    regParam=0.05,
    elasticNetParam=0.0,  # 0.0 = L2 regularization
    standardization=True
)

logistic_pipeline = Pipeline(
    stages=indexers + [assembler, logistic_regression]
)

logistic_start = time.perf_counter()

logistic_model = logistic_pipeline.fit(train_data)

print(
    f"Logistic Regression training time: "
    f"{time.perf_counter() - logistic_start:.1f}s"
)

Logistic Regression training time: 315.0s


In [ ]:


logistic_predictions = (
    logistic_model.transform(test_data)
    .select(
        "fraud_label",
        "prediction",
        "rawPrediction",
        "probability"
    )
    .withColumn(
        "fraud_probability",
        vector_to_array("probability")[1]
    )
    .persist(StorageLevel.DISK_ONLY)
)

logistic_test_count = logistic_predictions.count()

logistic_roc_auc = BinaryClassificationEvaluator(
    labelCol="fraud_label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
).evaluate(logistic_predictions)

logistic_pr_auc = BinaryClassificationEvaluator(
    labelCol="fraud_label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderPR"
).evaluate(logistic_predictions)

logistic_cm = logistic_predictions.agg(
    F.sum(
        F.when(
            (F.col("fraud_label") == 1.0) &
            (F.col("prediction") == 1.0),
            1
        ).otherwise(0)
    ).alias("TP"),

    F.sum(
        F.when(
            (F.col("fraud_label") == 0.0) &
            (F.col("prediction") == 1.0),
            1
        ).otherwise(0)
    ).alias("FP"),

    F.sum(
        F.when(
            (F.col("fraud_label") == 0.0) &
            (F.col("prediction") == 0.0),
            1
        ).otherwise(0)
    ).alias("TN"),

    F.sum(
        F.when(
            (F.col("fraud_label") == 1.0) &
            (F.col("prediction") == 0.0),
            1
        ).otherwise(0)
    ).alias("FN")
).first()

TP, FP, TN, FN = [
    int(logistic_cm[column] or 0)
    for column in ["TP", "FP", "TN", "FN"]
]

logistic_precision = TP / (TP + FP) if TP + FP else 0.0
logistic_recall = TP / (TP + FN) if TP + FN else 0.0
logistic_f1 = (
    2 * logistic_precision * logistic_recall /
    (logistic_precision + logistic_recall)
    if logistic_precision + logistic_recall else 0.0
)

print(f"Test rows: {logistic_test_count:,}")
print(f"ROC-AUC: {logistic_roc_auc:.4f}")
print(f"PR-AUC: {logistic_pr_auc:.4f}")
print(f"TP={TP:,}  FP={FP:,}  TN={TN:,}  FN={FN:,}")
print(f"Fraud precision: {logistic_precision:.4f}")
print(f"Fraud recall: {logistic_recall:.4f}")
print(f"Fraud F1-score: {logistic_f1:.4f}")

Test rows: 4,878,909
ROC-AUC: 0.8134
PR-AUC: 0.0097
TP=3,881  FP=694,693  TN=4,178,336  FN=1,999
Fraud precision: 0.0056
Fraud recall: 0.6600
Fraud F1-score: 0.0110


### Compare Logistic regression threshhold


In [ ]:
logistic_threshold_rows = []

for threshold in [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]:
    threshold_counts = logistic_predictions.agg(
        F.sum(
            F.when(
                (F.col("fraud_label") == 1.0) &
                (F.col("fraud_probability") >= threshold),
                1
            ).otherwise(0)
        ).alias("TP"),

        F.sum(
            F.when(
                (F.col("fraud_label") == 0.0) &
                (F.col("fraud_probability") >= threshold),
                1
            ).otherwise(0)
        ).alias("FP"),

        F.sum(
            F.when(
                (F.col("fraud_label") == 1.0) &
                (F.col("fraud_probability") < threshold),
                1
            ).otherwise(0)
        ).alias("FN")
    ).first()

    tp = int(threshold_counts["TP"] or 0)
    fp = int(threshold_counts["FP"] or 0)
    fn = int(threshold_counts["FN"] or 0)

    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0.0
    )

    logistic_threshold_rows.append(
        (threshold, precision, recall, f1)
    )

logistic_threshold_results = spark.createDataFrame(
    logistic_threshold_rows,
    ["threshold", "precision", "recall", "f1"]
)

logistic_threshold_results.orderBy(
    F.desc("f1")
).show(truncate=False)

+---------+---------------------+--------------------+---------------------+
|threshold|precision            |recall              |f1                   |
+---------+---------------------+--------------------+---------------------+
|0.9      |0.02825303917154435  |0.042687074829931976|0.03400162557572474  |
|0.8      |0.014910789066267353 |0.19982993197278912 |0.027750879761932877 |
|0.7      |0.006806743260681048 |0.5579931972789116  |0.013449422220035989 |
|0.6      |0.0063942369327331895|0.6192176870748299  |0.01265776578787726  |
|0.5      |0.005555603271807997 |0.6600340136054422  |0.011018462525587192 |
|0.4      |0.002731239749882649 |0.801530612244898   |0.005443929146909852 |
|0.3      |0.0019813147414397404|0.8974489795918368  |0.0039539003837388825|
|0.2      |0.0012291275599001933|0.9909863945578231  |0.0024552099052782797|
|0.1      |0.0012048731244203202|0.9994897959183674  |0.0024068448258774515|
+---------+---------------------+--------------------+---------------------+

In [ ]:
comparison_rows = [
    (
        "XGBoost",
        float(roc_auc),
        float(pr_auc),
        float(precision),
        float(recall),
        float(f1)
    ),
    (
        "Logistic Regression",
        float(logistic_roc_auc),
        float(logistic_pr_auc),
        float(logistic_precision),
        float(logistic_recall),
        float(logistic_f1)
    )
]

comparison_df = spark.createDataFrame(
    comparison_rows,
    [
        "model",
        "roc_auc",
        "pr_auc",
        "precision",
        "recall",
        "f1"
    ]
)

comparison_df.orderBy(
    F.desc("pr_auc")
).show(truncate=False)

+-------------------+------------------+-------------------+--------------------+--------------------+--------------------+
|model              |roc_auc           |pr_auc             |precision           |recall              |f1                  |
+-------------------+------------------+-------------------+--------------------+--------------------+--------------------+
|XGBoost            |0.9788162871719468|0.39890348755337335|0.02825303917154435 |0.042687074829931976|0.03400162557572474 |
|Logistic Regression|0.8133714982281977|0.00969267494482051|0.005555603271807997|0.6600340136054422  |0.011018462525587192|
+-------------------+------------------+-------------------+--------------------+--------------------+--------------------+



Conclusion:

- In full dataset, XGboost perform better than logistic regression
- Due to Unbalanced Sampling Simple sampling techniques would not work efficiently. SMOTE (Synthetic Minority Over-sampling Technique) usefull to reduce this limitation.
- In future, Deep learning techniques may gives more better result with extra features.
